# Fold NLP Model v3 - Multi-Head DistilBERT (EDA + Training)

Trains on `eda_dataset_v3.csv` with three classification heads:

- **Category** (10 classes): food, shopping, travel, etc.
- **Payment Method** (cash / upi / card / unknown)
- **Bank Account** (hdfc, sbi, slice, etc. + unknown)

This notebook is split into two halves:
1. **EDA & Cleaning** - explore the raw dataset, find real data-quality defects, and clean them.
2. **Training** - encode labels, tokenize, train the multi-head DistilBERT model.

In [ ]:
!pip install transformers datasets evaluate

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from datasets import Dataset
import json
import re

## Part 1: Load & Inspect the Raw Dataset

Start by loading the raw file and getting a high-level view: shape, columns, dtypes.

In [ ]:
df = pd.read_csv('eda_dataset_v3.csv')
print(f'Loaded {len(df)} rows')
print(f'Columns: {list(df.columns)}')
df.info()
df.head(10)

## Part 2: Missing Values

Missing data rarely shows up as just `NaN`. Real-world CSVs use **multiple sentinels**:
empty string, `NA`, `null`, `nan`, `None`, `--`. A naive `.isnull()` check misses all of them.

Let us audit every column against these sentinels.

In [ ]:
SENTINELS = ['', 'NA', 'null', 'nan', 'None', '--']

def count_missing(series):
    s = series.astype(str).str.strip()
    null_like = s.isna().sum() + (s == 'nan').sum()
    sentinel = s.isin(SENTINELS).sum()
    return int(null_like + sentinel)

missing_report = {col: count_missing(df[col]) for col in df.columns}
print('Missing values per column (NaN + string sentinels):')
for col, n in missing_report.items():
    print(f'  {col:20s}: {n}')

## Part 3: Category Canonicalization

The `category` column should have **exactly 10** canonical values. Let us see what is actually there -
typos, synonyms, mixed case, and whitespace all create extra categories that do not exist.

In [ ]:
print(f"Distinct category values: {df['category'].nunique(dropna=False)}")
print('\nFull value counts:')
print(df['category'].value_counts(dropna=False).to_string())

CANONICAL_CATEGORIES = {
    'food', 'shopping', 'travel', 'entertainment', 'healthcare',
    'education', 'emi', 'utilities', 'investment', 'friends'
}
extra = set(df['category'].dropna().str.strip().str.lower().unique()) - CANONICAL_CATEGORIES
print(f'\nNon-canonical values found: {sorted(extra)}')

## Part 4: Duplicates

Duplicate rows waste training capacity and inflate metrics. Check both **exact** duplicates and
**near-duplicates** (same content, different surrounding whitespace).

In [ ]:
exact_dups = df.duplicated().sum()
print(f'Exact duplicate rows: {exact_dups}')

near = df['text'].astype(str).str.strip().duplicated().sum()
print(f'Near-duplicate texts (after strip): {near}')

## Part 5: Amount Anomalies

The `amount` column should be a positive number. Look for:
- non-numeric / string-form amounts (`1,234.00`)
- negatives (refunds)
- zeros and absurd outliers

In [ ]:
df['_amount_num'] = pd.to_numeric(df['amount'].astype(str).str.replace(',', ''), errors='coerce')

print('Amount parse failures (non-numeric):', df['_amount_num'].isna().sum())
print('Negative amounts:', (df['_amount_num'] < 0).sum())
print('Zero amounts:', (df['_amount_num'] == 0).sum())
print('\nAmount distribution (numeric only):')
print(df['_amount_num'].describe())
print('\nLargest 5 amounts:')
print(df['_amount_num'].dropna().nlargest(5))

## Part 6: Encoding Artifacts

Some rows contain mojibake (e.g. a Windows-1252 misdecode of the rupee sign).
These are invisible in some editors but break tokenization.

In [ ]:
mojibake = df['text'].astype(str).str.contains(r'[\x80-\x9f\u00c2\u00e2]', regex=True, na=False)
print(f'Rows with likely mojibake/encoding artifacts: {mojibake.sum()}')
print(df[mojibake]['text'].head().to_string())

## Part 7: Whitespace & Case Noise

Leading/trailing/double spaces and inconsistent casing (`Food`, `FOOD`, ` food `)
turn one category into several from the model perspective.

In [ ]:
ws = df['text'].astype(str).apply(lambda x: x != x.strip() or '  ' in x)
print(f'Rows with whitespace noise in text: {ws.sum()}')
print('Payment method casing variants:', df['payment_method'].astype(str).str.strip().value_counts().to_dict())

## Part 8: Cleaning Pipeline

Now we fix every defect found above, step by step. Each step is a deliberate transformation.

In [ ]:
clean = df.copy()
print(f'Before cleaning: {len(clean)} rows')

# 1) Drop rows missing the critical fields we cannot impute
clean = clean.dropna(subset=['text', 'category'])
clean = clean[clean['text'].astype(str).str.strip() != '']
clean = clean[clean['category'].astype(str).str.strip() != '']
print(f'After dropping missing text/category: {len(clean)} rows')

# 2) Strip whitespace + collapse double spaces in text
clean['text'] = clean['text'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# 3) Normalize payment_method: strip + lowercase + canonical
VALID_METHODS = {'cash', 'upi', 'card'}
clean['payment_method'] = clean['payment_method'].fillna('unknown').astype(str).str.strip().str.lower()
clean.loc[~clean['payment_method'].isin(VALID_METHODS), 'payment_method'] = 'unknown'

# 4) Canonicalize category: strip + lowercase, then map synonyms/typos
clean['category'] = clean['category'].astype(str).str.strip().str.lower()
CATEGORY_MAP = {
    'foods':'food','dining':'food','meal':'food','restuarant':'food','grocery':'food',
    'shoping':'shopping','purchase':'shopping','retail':'shopping',
    'travell':'travel','transport':'travel','commute':'travel','fuel':'travel','cab':'travel',
    'entrtainment':'entertainment','movies':'entertainment','ott':'entertainment','leisure':'entertainment',
    'healthcar':'healthcare','health':'healthcare','medical':'healthcare','pharmacy':'healthcare','hospital':'healthcare',
    'eduation':'education','tution':'education','school':'education','study':'education',
    'loan':'emi','installment':'emi','loans':'emi',
    'utilites':'utilities','bills':'utilities','recharge':'utilities','utility':'utilities',
    'invstment':'investment','invest':'investment','savings':'investment','mutual fund':'investment',
    'freind':'friends','p2p':'friends','transfer':'friends','peer':'friends',
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)
clean = clean[clean['category'].isin(CANONICAL_CATEGORIES)].copy()
print(f'After canonicalizing categories: {len(clean)} rows')

# 5) Parse amount: handle commas, negatives, outliers
clean['amount'] = pd.to_numeric(clean['amount'].astype(str).str.replace(',', ''), errors='coerce')
clean = clean.dropna(subset=['amount'])
clean = clean[clean['amount'] > 0]                       # drop zeros/negatives
clean = clean[clean['amount'] < 1000000]                 # drop absurd outliers
clean['amount'] = clean['amount'].astype(float)
print(f'After amount validation: {len(clean)} rows')

# 6) Drop duplicate rows (exact + near)
clean = clean.drop_duplicates(subset=['text', 'amount', 'category'], keep='first')
print(f'After dropping duplicates: {len(clean)} rows')

# 7) Canonicalize bank_account: map display names back to canonical keys
BANK_DISPLAY = {
    'hdfc':'hdfc','sbi':'sbi','icici':'icici','axis':'axis','kotak':'kotak','pnb':'pnb',
    'bob':'bob','yes bank':'yes bank','idfc':'idfc','indusind':'indusind','canara':'canara',
    'union bank':'union bank','federal bank':'federal bank','rbl':'rbl','bandhan':'bandhan',
    'slice':'slice','jupiter':'jupiter','fi':'fi','niyo':'niyo',
}
clean['bank_account'] = clean['bank_account'].fillna('unknown').astype(str).str.strip().str.lower()
clean['bank_account'] = clean['bank_account'].replace(BANK_DISPLAY)
clean.loc[clean['bank_account'] == '', 'bank_account'] = 'unknown'

# 8) Clean mojibake from text
clean['text'] = clean['text'].str.replace(r'[\x80-\x9f\u00c2\u00e2]', '', regex=True)

clean = clean.drop(columns=['_amount_num']).reset_index(drop=True)
print(f'\nFINAL cleaned dataset: {len(clean)} rows')
clean.head()

## Part 9: Before vs After - Did EDA Do Anything?

Compare the raw and cleaned datasets. A good cleaning step *changes* the data.

In [ ]:
print(f'Raw rows:      {len(df)}')
print(f'Cleaned rows:  {len(clean)}')
print(f'Rows removed:  {len(df) - len(clean)}')
print(f'\nDistinct categories  raw: {df["category"].nunique(dropna=False)}  ->  cleaned: {clean["category"].nunique()}')
print(f'\nCleaned category distribution:')
print(clean['category'].value_counts().to_string())
print(f'\nCleaned payment method distribution:')
print(clean['payment_method'].value_counts().to_string())
print(f'\nCleaned text source distribution:')
print(clean['text_source'].value_counts().to_string())

## Part 10: Encode Labels

Now that the data is clean, encode the three target columns into integer labels.

In [ ]:
le_cat = LabelEncoder()
le_method = LabelEncoder()
le_bank = LabelEncoder()

clean['label_cat'] = le_cat.fit_transform(clean['category'])
clean['label_method'] = le_method.fit_transform(clean['payment_method'])
clean['label_bank'] = le_bank.fit_transform(clean['bank_account'])

num_cat = len(le_cat.classes_)
num_method = len(le_method.classes_)
num_bank = len(le_bank.classes_)

print(f'Category classes ({num_cat}): {list(le_cat.classes_)}')
print(f'Method classes ({num_method}): {list(le_method.classes_)}')
print(f'Bank classes ({num_bank}): {list(le_bank.classes_)}')

label_maps = {
    'category': {int(i): str(c) for i, c in enumerate(le_cat.classes_)},
    'payment_method': {int(i): str(c) for i, c in enumerate(le_method.classes_)},
    'bank_account': {int(i): str(c) for i, c in enumerate(le_bank.classes_)},
}
with open('label_maps_v3.json', 'w') as f:
    json.dump(label_maps, f, indent=2)
print('\nSaved label_maps_v3.json')

In [ ]:
train_df, val_df = train_test_split(
    clean[['text', 'label_cat', 'label_method', 'label_bank']],
    test_size=0.15, random_state=42, stratify=clean['category']
)
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_val = val_dataset.map(tokenize_fn, batched=True)

In [ ]:
class MultiHeadDistilBERT(nn.Module):
    """DistilBERT encoder with three classification heads."""

    def __init__(self, num_cat, num_method, num_bank):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('distilbert-base-uncased')
        hidden = self.encoder.config.hidden_size  # 768
        self.dropout = nn.Dropout(0.1)
        self.head_cat = nn.Linear(hidden, num_cat)
        self.head_method = nn.Linear(hidden, num_method)
        self.head_bank = nn.Linear(hidden, num_bank)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None,
                label_cat=None, label_method=None, label_bank=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits_cat = self.head_cat(cls_output)
        logits_method = self.head_method(cls_output)
        logits_bank = self.head_bank(cls_output)

        loss = None
        if label_cat is not None:
            loss_cat = self.loss_fn(logits_cat, label_cat)
            loss_method = self.loss_fn(logits_method, label_method)
            loss_bank = self.loss_fn(logits_bank, label_bank)
            loss = loss_cat + 0.5 * loss_method + 0.5 * loss_bank

        return {
            'loss': loss,
            'logits_cat': logits_cat,
            'logits_method': logits_method,
            'logits_bank': logits_bank,
        }

model = MultiHeadDistilBERT(num_cat, num_method, num_bank)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model on {device}')
print(f'Heads: cat={num_cat}, method={num_method}, bank={num_bank}')

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    input_ids = torch.tensor([b['input_ids'] for b in batch])
    attention_mask = torch.tensor([b['attention_mask'] for b in batch])
    label_cat = torch.tensor([b['label_cat'] for b in batch])
    label_method = torch.tensor([b['label_method'] for b in batch])
    label_bank = torch.tensor([b['label_bank'] for b in batch])
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'label_cat': label_cat,
        'label_method': label_method,
        'label_bank': label_bank,
    }

train_loader = DataLoader(tokenized_train, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(tokenized_val, batch_size=64, shuffle=False, collate_fn=collate_fn)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR

NUM_EPOCHS = 4
LR = 2e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out['loss']
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if step % 100 == 0:
            print(f'  Epoch {epoch+1}/{NUM_EPOCHS} step {step}/{len(train_loader)} loss={loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)

    model.eval()
    correct_cat = correct_method = correct_bank = total = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            pred_cat = out['logits_cat'].argmax(dim=1)
            pred_method = out['logits_method'].argmax(dim=1)
            pred_bank = out['logits_bank'].argmax(dim=1)
            correct_cat += (pred_cat == batch['label_cat']).sum().item()
            correct_method += (pred_method == batch['label_method']).sum().item()
            correct_bank += (pred_bank == batch['label_bank']).sum().item()
            total += len(batch['label_cat'])

    print(f'Epoch {epoch+1}: loss={avg_loss:.4f} | '
          f'cat_acc={correct_cat/total:.4f} | '
          f'method_acc={correct_method/total:.4f} | '
          f'bank_acc={correct_bank/total:.4f}')

In [ ]:
import os, shutil

SAVE_DIR = 'my_finetuned_distilbert_v3'
os.makedirs(SAVE_DIR, exist_ok=True)

model.encoder.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

torch.save({
    'head_cat': model.head_cat.state_dict(),
    'head_method': model.head_method.state_dict(),
    'head_bank': model.head_bank.state_dict(),
    'num_cat': num_cat,
    'num_method': num_method,
    'num_bank': num_bank,
}, os.path.join(SAVE_DIR, 'heads.pt'))

shutil.copy('label_maps_v3.json', os.path.join(SAVE_DIR, 'label_maps_v3.json'))

print(f'Saved to {SAVE_DIR}/')
print('Files:', os.listdir(SAVE_DIR))
shutil.make_archive('my_model_v3', 'zip', '.', SAVE_DIR)
print('Zipped to my_model_v3.zip')

In [ ]:
model.eval()
test_texts = [
    'paid 230 for electricity via slice UPI',
    'Swiggy se pizza mangwaya 450 rupaye gpay se',
    'Amazon purchase 2000 hdfc card',
    'sent 500 to Rahul on PhonePe',
    'petrol 1500 cash',
]

for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = model(**inputs)
    cat = le_cat.classes_[out['logits_cat'].argmax(dim=1).item()]
    method = le_method.classes_[out['logits_method'].argmax(dim=1).item()]
    bank = le_bank.classes_[out['logits_bank'].argmax(dim=1).item()]
    print(f'{text}')
    print(f'  -> category={cat}, method={method}, bank={bank}\n')